# State

In [26]:
from typing import TypedDict

class State(TypedDict):
    operation: str
    processed_value: int

obj = State({'first_value':1,"second_value":2,"answer":3})
obj['first_value']

1

In [ ]:
# from pydantic import BaseModel

# class State(BaseModel):
#     first_value: int
#     second_value: int
#     answer: int

# Graph

In [27]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)

# Node

In [28]:
def additional_node(state: State):
    value = state.get('processed_value')
    out = value+2
    return {
        'processed_value':out
    }

def subraction_node(state: State):
    value = state.get('processed_value')
    out = value-2
    return {
        'processed_value':out
    }

def multiply_node(state: State):
    value = state.get('processed_value')
    out = value*2
    return {
        'processed_value':out
    }

graph.add_node("addition",additional_node)
graph.add_node("sub",subraction_node)
graph.add_node("multiply", multiply_node)
    

# Edge

In [ ]:
graph.add_edge(START, "addition")
graph.add_edge("addition","sub")
graph.add_edge("sub","multiply")
graph.add_edge("multiply",END)

# COMPILE

In [23]:
agent = graph.compile()

# Invoking

In [24]:
response = agent.invoke({'processed_value': 4})
print(response.get('processed_value'))

8


In [25]:
from PIL import Image
import io

img_bytes = agent.get_graph().draw_mermaid_png()
image = Image.open(io.BytesIO(img_bytes))
image.show()

# Conditional edges

In [29]:
def router_function(state: State):
    operation = state.get('operation')
    if operation == 'add':
        return 'add'
    elif operation == 'sub':
        return 'sub'
    elif operation == 'mul':
        return 'mul'

graph.add_conditional_edges(
    START,
    router_function,
    {
        'add':'addition',
        'sub':'sub',
        'mul':'multiply'
    }
)

graph.add_edge('addition',END)
graph.add_edge('sub',END)
graph.add_edge('multiply',END)

In [32]:
agent = graph.compile()

response = agent.invoke({'operation':'mul','processed_value':6})
print(response['processed_value'])

12


In [33]:
from PIL import Image
import io

img_bytes = agent.get_graph().draw_mermaid_png()
image = Image.open(io.BytesIO(img_bytes))
image.show()

In [ ]:
[
    {'role':'ai','content':'hello, how can i help you today'}
]

# AI agent

In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

response = llm.invoke('hii')

def chat_llm(state: State):
    response = llm.invoke(state['messages'])
    return {'messages':[response]}

graph = StateGraph(State)

graph.add_node('llm_node', chat_llm)

graph.add_edge(START, 'llm_node')
graph.add_edge('llm_node', END)

agent = graph.compile()

response = agent.invoke({'messages':'hi'})
print(response)

{'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='ebe077ab-cb27-42e3-aa9d-903a59356972'), AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a11172-c82e-7580-b60d-495d04354540-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 39, 'total_tokens': 41, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 29}})]}
